# CLEVER retrieval + repair: get results on Colab

**Before anything:** Runtime → Change runtime type → **T4 GPU**.

Run cells top to bottom. Logs, LLM cache and results are saved in Google Drive (`MyDrive/clever-rr/`).
After a disconnect: run cells 0–4 again, then the experiment cell you were on. Finished problems are skipped and model replies are cached.

## 0. GPU check
Must print `Tesla T4`. If it errors, change the runtime type to T4 GPU first.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/Aditi-Nandiraju/Clever.git /content/Clever || git -C /content/Clever pull -q
%cd /content/Clever

SAVE = '/content/drive/MyDrive/clever-rr'
for d in ['logs', 'cache', 'results']:
    os.makedirs(f'{SAVE}/{d}', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf {d}
        os.symlink(f'{SAVE}/{d}', d)
print('outputs go to', SAVE)

## 2. Python packages

In [ ]:
!pip install -q -r requirements.txt

## 3. Lean 4 + Mathlib (~10–20 min on a fresh runtime)
Only `Imports.AllImports` is built: every problem file imports just that, so building all 161 problems is not needed.
The last line compiles a tiny test file. It must print `4` and no errors, otherwise every attempt will fail with `impl`.

In [ ]:
import os
if not os.path.exists(os.path.expanduser('~/.elan/bin/lake')):
    !curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
os.environ['PATH'] = os.path.expanduser('~/.elan/bin') + ':' + os.environ['PATH']

from clever_bench.benchmark import get_clever_lean_project_path
LEAN_DIR = str(get_clever_lean_project_path())
!cd {LEAN_DIR} && lake exe cache get
!cd {LEAN_DIR} && lake build Imports.AllImports

# Lean sanity check
os.makedirs(f'{LEAN_DIR}/temp', exist_ok=True)
open(f'{LEAN_DIR}/temp/sanity.lean', 'w').write('import Imports.AllImports\n#eval (2 + 2 : Nat)\nexample : (1 : Nat) + 1 = 2 := by norm_num\n')
!cd {LEAN_DIR} && lake lean temp/sanity.lean

## 4. Ollama on the GPU
The warnings `systemd is not running` and `Unable to detect NVIDIA/AMD GPU` are harmless.
The last line must show **100% GPU** under PROCESSOR.

In [ ]:
import subprocess, time, shutil
if not shutil.which('ollama'):
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'a'), stderr=subprocess.STDOUT)
time.sleep(5)
open('/content/Modelfile', 'w').write('FROM qwen2.5:7b\nPARAMETER num_ctx 32768\n')
!ollama pull qwen2.5:7b && ollama create qwen2.5-32k -f /content/Modelfile
!ollama run qwen2.5-32k "hi" > /dev/null; ollama ps

## 5. Retrieval corpus (~2 min)

In [ ]:
!python corpus.py

## 6. Quick test: 3 dev problems (~10–25 min)
Stages: `parse` = bad answer format, `impl` = implementation fails to compile/tests, `proof` = implementation OK but proof fails, `ok PROVED` = success.

In [ ]:
!python run_experiment.py --configs C1_baseline --split dev --limit 3 --seeds 0

### Why did attempts fail?
Real Lean errors about the code (`unknown identifier`, `type mismatch`, failed tests) = the model is weak, that is a valid result, carry on.
`Timeout`, `object file ... does not exist` or errors mentioning `Imports` = Lean setup problem: rerun cell 3, then `!rm logs/C1_baseline_s0_dev.jsonl` and rerun cell 6 (model replies are cached).

In [ ]:
import json, glob
for f in sorted(glob.glob('logs/*_dev.jsonl')):
    print('=====', f)
    for l in open(f):
        r = json.loads(l)
        print(f"p{r['pid']} a{r['attempt']} {r['stage']:<6} llm {r['sec_llm']}s  total {r['sec_total']}s")
        print('   ', (r['feedback'] or '')[:400].replace('\n', '\n    '), '\n')

## 7. First results: baseline vs repair on 10 dev problems (~1–2 h)

In [ ]:
!python run_experiment.py --configs C1_baseline C2_repair --split dev --limit 10 --seeds 0
!python analyze.py --split dev

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/dev/summary.csv'))
display(Image('results/dev/curve.png'))
display(Image('results/dev/errors.png'))

## 8. Optional: add retrieval (C3, C4) on the same 10 dev problems

In [ ]:
!python run_experiment.py --configs C3_retrieval C4_both --split dev --limit 10 --seeds 0
!python analyze.py --split dev
display(pd.read_csv('results/dev/summary.csv'))

## 9. Final numbers: 40 test problems
Freeze `config.py` and `prompts.py` first. This takes several Colab sessions; rerun the cell after each disconnect.
Results: `MyDrive/clever-rr/results/test/`.

In [ ]:
!python run_experiment.py --configs C1_baseline C2_repair --seeds 0
!python run_experiment.py --configs C3_retrieval C4_both --seeds 0
!python analyze.py
display(pd.read_csv('results/test/summary.csv'))